<a href="https://colab.research.google.com/github/anu04596/Generative-AI/blob/main/Practical_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
# Step 1: Imports
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist
import numpy as np
import keras.ops as ops

# Step 2: Load Data
(x_train, _), (x_test, _) = mnist.load_data()

x_train = x_train.astype("float32") / 255.
x_test = x_test.astype("float32") / 255.

x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

# Step 3: Latent dimension
latent_dim = 2

# Step 4: Encoder
encoder_inputs = layers.Input(shape=(784,))
x = layers.Dense(64, activation="relu")(encoder_inputs)
z_mean = layers.Dense(latent_dim)(x)
z_log_var = layers.Dense(latent_dim)(x)

encoder = tf.keras.Model(encoder_inputs, [z_mean, z_log_var], name="encoder")

# Step 5: Decoder
decoder_inputs = layers.Input(shape=(latent_dim,))
x = layers.Dense(64, activation="relu")(decoder_inputs)
decoder_outputs = layers.Dense(784, activation="sigmoid")(x)

decoder = tf.keras.Model(decoder_inputs, decoder_outputs, name="decoder")

# Step 6: VAE Model (Subclassing tf.keras.Model)
class VAE(tf.keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder

    def call(self, inputs):
        z_mean, z_log_var = self.encoder(inputs)
        epsilon = tf.random.normal(shape=tf.shape(z_mean))
        z = z_mean + ops.exp(0.5 * z_log_var) * epsilon
        reconstruction = self.decoder(z)

        reconstruction_loss = ops.mean(
            tf.keras.losses.binary_crossentropy(inputs, reconstruction)
        ) * 784

        kl_loss = -0.5 * ops.mean(
            1 + z_log_var - ops.square(z_mean) - ops.exp(z_log_var)
        )

        self.add_loss(reconstruction_loss + kl_loss)
        return reconstruction

# Instantiate and Compile the VAE
vae = VAE(encoder, decoder)
vae.compile(optimizer="adam")

# Step 7: Train
vae.fit(x_train, x_train,
        epochs=5,
        batch_size=128,
        validation_data=(x_test, x_test))

Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 221.1421 - val_loss: 186.4925
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 183.0988 - val_loss: 178.3549
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 175.7020 - val_loss: 171.5981
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 170.6177 - val_loss: 168.5006
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 168.2188 - val_loss: 166.7364
